# Stage 5 — Retention Intelligence Room Walkthrough

**Chapters 10–11** of *Mastering Agentic AI for Customer Journey Marketing*
by Pushparajan Ramar.

**Framework:** AutoGen + MCP

This notebook walks through the Retention Intelligence Room step by step:
1. Tool exploration — inspect mock data and tool outputs
2. Trigger evaluation — test the five trigger conditions
3. Individual agent testing — run each agent in isolation
4. Full room execution — orchestrate all agents in a group chat

## 0. Setup

In [ ]:
import os
import sys
import json

# Ensure mock mode is on
os.environ["USE_MOCK_APIS"] = "true"

# Add stage5 root to path
STAGE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if STAGE_DIR not in sys.path:
    sys.path.insert(0, STAGE_DIR)

print(f"Stage directory: {STAGE_DIR}")
print(f"USE_MOCK_APIS:  {os.getenv('USE_MOCK_APIS')}")

## 1. Tool Exploration

Let's explore the tools individually to understand the data they provide.

### 1.1 Churn Risk Tools

In [ ]:
from tools.churn_risk_tools import (
    get_churn_risk_score,
    predict_churn_probability,
    get_similar_churned_customers,
    get_successful_retention_plays,
)

# Get churn risk profile for CUST-001 (critical risk)
risk_profile = json.loads(get_churn_risk_score("CUST-001"))
print("=== Churn Risk Profile — CUST-001 ===")
print(json.dumps(risk_profile, indent=2))

In [ ]:
# Predict churn probability at 30, 60, 90 day horizons
for days in [30, 60, 90]:
    prediction = json.loads(predict_churn_probability("CUST-001", days))
    print(f"\n{days}-day churn probability: {prediction['probability']:.0%}")
    print(f"  Confidence interval: {prediction['confidence_interval']}")

In [ ]:
# Find similar churned customers
similar = json.loads(get_similar_churned_customers("critical", "usage_gap"))
print("=== Similar Churned Customers ===")
print(json.dumps(similar, indent=2))

In [ ]:
# Look up successful retention plays for usage_gap driver
plays = json.loads(get_successful_retention_plays("usage_gap"))
print("=== Successful Retention Plays (usage_gap) ===")
print(json.dumps(plays, indent=2))

### 1.2 NPS and Sentiment Tools

In [ ]:
from tools.nps_sentiment_tools import (
    get_nps_score,
    get_sentiment_trend,
    get_support_ticket_sentiment,
)

# NPS score
nps = json.loads(get_nps_score("CUST-001"))
print("=== NPS Score — CUST-001 ===")
print(f"Current: {nps['current_nps']}  Previous: {nps['previous_nps']}  Trend: {nps['nps_trend']}")
print(f"Category: {nps['category']}")
print(f"Verbatim: {nps['verbatim']}")

In [ ]:
# Sentiment trend
sentiment = json.loads(get_sentiment_trend("CUST-001", 90))
print("=== Sentiment Trend — CUST-001 (90 days) ===")
print(f"Overall: {sentiment['overall_sentiment']} (score: {sentiment['sentiment_score']})")
print(f"Direction: {sentiment['trend_direction']}")
print(f"Key themes: {sentiment['key_themes']}")

In [ ]:
# Support ticket sentiment
tickets = json.loads(get_support_ticket_sentiment("CUST-001"))
print("=== Support Ticket Sentiment — CUST-001 ===")
print(f"Total: {tickets['total_tickets_30d']}  Negative: {tickets['negative_tickets']}")
print(f"Consecutive negative: {tickets['consecutive_negative']}")
print(f"Escalated: {tickets['escalated']}")
for t in tickets['tickets']:
    print(f"  [{t['sentiment']}] {t['id']}: {t['subject']}")

### 1.3 Contract and Expansion Tools

In [ ]:
from tools.expansion_revenue_tools import (
    get_contract_details,
    calculate_retention_offer,
    get_expansion_opportunities,
)

# Contract details
contract = json.loads(get_contract_details("CUST-001"))
print("=== Contract — CUST-001 ===")
print(f"Plan: {contract['plan']}  ARR: ${contract['arr']:,}")
print(f"Renewal in: {contract['days_to_renewal']} days")
print(f"Seats: {contract['seats_active']}/{contract['seats_licensed']} active")

In [ ]:
# Retention offer
offer = json.loads(calculate_retention_offer("CUST-001", "critical"))
print("=== Retention Offer — CUST-001 (critical) ===")
print(json.dumps(offer, indent=2))

In [ ]:
# Expansion opportunities
expansion = json.loads(get_expansion_opportunities("CUST-001"))
print("=== Expansion Opportunities — CUST-001 ===")
print(json.dumps(expansion, indent=2))

### 1.4 Win-Back Tools

In [ ]:
from tools.win_back_tools import (
    generate_win_back_sequence,
    get_competitive_counter_offer,
)

# Win-back sequence
sequence = json.loads(generate_win_back_sequence("CUST-001", "usage_gap"))
print("=== Win-Back Sequence — CUST-001 (usage_gap) ===")
print(f"Name: {sequence['sequence_name']}")
print(f"Duration: {sequence['duration_days']} days, {sequence['total_touches']} touches")
for step in sequence['steps']:
    print(f"  Day {step['day']:2d} | {step['channel']:8s} | {step['action']} ({step['owner']})")

In [ ]:
# Competitive counter-offer
counter = json.loads(get_competitive_counter_offer("CompetitorX"))
print("=== Competitive Counter — CompetitorX ===")
print(f"Positioning: {counter['positioning']}")
print(f"Switching cost: ${counter['switching_cost_estimate']:,}")
print(f"Migration time: {counter['migration_time_weeks']} weeks")
print("\nOur advantages:")
for adv in counter['our_advantages']:
    print(f"  + {adv}")

## 2. Trigger Evaluation

Test the five trigger conditions against each customer.

In [ ]:
from groupchats.retention_intelligence_room import evaluate_triggers

for cust_id in ["CUST-001", "CUST-002", "CUST-003"]:
    result = evaluate_triggers(cust_id)
    print(f"\n{'='*50}")
    print(f"Customer: {cust_id}")
    print(f"Should activate: {result['should_activate']}")
    print(f"Triggers fired ({result['trigger_count']}):")
    for t in result['triggers_fired']:
        print(f"  - {t}")
    if not result['triggers_fired']:
        print("  (none)")

## 3. Individual Agent Testing

Test each agent independently. Requires `OPENAI_API_KEY` to be set.

**Note:** If you do not have an API key, you can skip this section and
proceed to Section 4 to review the expected group chat flow.

In [ ]:
# Uncomment and set your API key to run agent tests
# os.environ["OPENAI_API_KEY"] = "sk-..."

HAS_API_KEY = bool(os.getenv("OPENAI_API_KEY"))
print(f"OpenAI API key available: {HAS_API_KEY}")
if not HAS_API_KEY:
    print("Set OPENAI_API_KEY to run agent tests. Skipping agent sections.")

In [ ]:
if HAS_API_KEY:
    from autogen_agentchat.messages import TextMessage
    from autogen_core import CancellationToken
    from agents.churn_analyst import create_churn_analyst

    analyst = create_churn_analyst()
    response = await analyst.on_messages(
        [TextMessage(content="Assess churn risk for customer CUST-001.", source="user")],
        cancellation_token=CancellationToken(),
    )
    print("=== Churn Analyst Response ===")
    print(response.chat_message.content)
else:
    print("Skipped — no API key.")

In [ ]:
if HAS_API_KEY:
    from agents.retention_strategist import create_retention_strategist

    strategist = create_retention_strategist()
    response = await strategist.on_messages(
        [TextMessage(
            content=(
                "CUST-002 is classified as high risk with competitor as primary driver. "
                "The competitor is CompetitorX. Recommend a retention strategy."
            ),
            source="user",
        )],
        cancellation_token=CancellationToken(),
    )
    print("=== Retention Strategist Response ===")
    print(response.chat_message.content)
else:
    print("Skipped — no API key.")

In [ ]:
if HAS_API_KEY:
    from agents.commercial_agent import create_commercial_agent

    commercial = create_commercial_agent()
    response = await commercial.on_messages(
        [TextMessage(
            content=(
                "Prepare a retention offer for CUST-001 classified as critical risk. "
                "Include contract details and expansion opportunities."
            ),
            source="user",
        )],
        cancellation_token=CancellationToken(),
    )
    print("=== Commercial Agent Response ===")
    print(response.chat_message.content)
else:
    print("Skipped — no API key.")

## 4. Full Retention Room Execution

Run the complete Retention Intelligence Room with all three agents
collaborating in a group chat.

Requires `OPENAI_API_KEY`.

In [ ]:
if HAS_API_KEY:
    from groupchats.retention_intelligence_room import run_retention_room

    result = await run_retention_room(
        customer_id="CUST-001",
        scenario_context=(
            "TechCorp Inc. has seen a dramatic usage decline. "
            "Active features dropped from 85% to 28%. NPS fell from 8 to 4. "
            "Contract renewal is in 7 days."
        ),
    )

    print(f"\n{'='*50}")
    print("Room activated:", result["room_activated"])
    print("Triggers fired:", result["triggers"]["triggers_fired"])
    if result.get("final_response"):
        print("\n--- Final Intervention Plan ---")
        print(result["final_response"])
else:
    print("Skipped — no API key. Set OPENAI_API_KEY to run the full room.")

## 5. Scenario Comparison

Compare trigger evaluations across all three demo scenarios.

In [ ]:
scenarios = [
    {"id": 1, "name": "Usage Decline", "customer_id": "CUST-001"},
    {"id": 2, "name": "Competitor Threat", "customer_id": "CUST-002"},
    {"id": 3, "name": "Renewal at Risk", "customer_id": "CUST-003"},
]

print(f"{'Scenario':<22} {'Triggers':>8}  {'Risk':<10}  Fired")
print("-" * 70)

for s in scenarios:
    trig = evaluate_triggers(s["customer_id"])
    risk = json.loads(get_churn_risk_score(s["customer_id"]))
    print(
        f"{s['name']:<22} {trig['trigger_count']:>8}  "
        f"{risk['risk_level']:<10}  {', '.join(trig['triggers_fired'])}"
    )

## Summary

This walkthrough demonstrated:

1. **Tool layer** — Four tool modules providing churn risk, NPS/sentiment,
   contract/expansion, and win-back capabilities with mock + real API fallback.

2. **Trigger system** — Five automated trigger conditions that activate the
   retention room when customer health signals cross thresholds.

3. **Agent specialisation** — Three agents (Churn Analyst, Retention
   Strategist, Commercial Agent) each with distinct tools and responsibilities.

4. **Group chat orchestration** — AutoGen RoundRobinGroupChat coordinates
   the agents to produce a consolidated intervention plan.

**Next steps:** Extend to Stage 6 (Advocacy) or integrate with real CRM,
helpdesk, and NPS platforms by setting `USE_MOCK_APIS=false`.